# Published detector on Conakry

Rußwurm et al. (2023) trained a deep model on MARIDA and other sets, with many hard negatives. Their weights are public, so this tests one question: does it stay off Guinea's tidal flats, where the other methods fail?

MariNeXt (Kikaki et al. 2024) is tested the same way in section 4.

Install the detector in its own environment, so that it does not change the versions the other scripts rely on. The next cell does that, once.

In [1]:
from pathlib import Path
import json
import os
import rasterio
import pandas as pd

os.environ["MPLBACKEND"] = "Agg"  # the detector runs outside this notebook
ROOT = Path(r"D:\Guinea_Methodology\plasticwaste_experiments")
%cd {ROOT}
(ROOT / "7_published_detector/outputs").mkdir(parents=True, exist_ok=True)
MDD = r"D:\guinea_data\mdd_env\Scripts\marinedebrisdetector"

D:\Guinea_Methodology\plasticwaste_experiments


C:\Users\ASUS\AppData\Roaming\Python\Python310\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
!python -m venv D:\guinea_data\mdd_env
!D:\guinea_data\mdd_env\Scripts\python -m pip install -q marinedebrisdetector
!{MDD} --help

You should consider upgrading via the 'D:\guinea_data\mdd_env\Scripts\python.exe -m pip install --upgrade pip' command.


usage: MarineDebrisDetector [-h]
                            [--model {unetpp,unetpp_no_label_refinement,unet}]
                            [--plot-qualitative] [--seed {1,2,3}]
                            [--device {cuda,cpu}] [--ensemble]
                            [--test-time-augmentation] [--download-accra]
                            [--download-durban]
                            [scene]

positional arguments:
  scene

options:
  -h, --help            show this help message and exit
  --model {unetpp,unetpp_no_label_refinement,unet}
  --plot-qualitative
  --seed {1,2,3}
  --device {cuda,cpu}
  --ensemble
  --test-time-augmentation
  --download-accra
  --download-durban


## 1. Download Conakry, two dates, about 600 MB each

In [3]:
# !python 7_published_detector/01_download_conakry.py
# with rasterio.open(ROOT / "7_published_detector/outputs/conakry_2024-01-11.tif") as s:
#     print("bands", s.count, "type", s.dtypes[0], "max of band 4", s.read(4, window=((0, 500), (0, 500))).max())

## 2. Run the detector

About five minutes a date on CPU. The detector's own command line tool leaves a grid of false detections along its window edges, so this script runs the same weights on overlapping windows.

In [4]:
!D:\guinea_data\mdd_env\Scripts\python 7_published_detector/run_detector.py

threshold stored with the weights 0.06387047469615936
conakry_2024-01-11.tif 0 of 120
conakry_2024-01-11.tif 20 of 120
conakry_2024-01-11.tif 40 of 120
conakry_2024-01-11.tif 60 of 120
conakry_2024-01-11.tif 80 of 120
conakry_2024-01-11.tif 100 of 120
conakry_2024-01-26.tif 0 of 120
conakry_2024-01-26.tif 20 of 120
conakry_2024-01-26.tif 40 of 120
conakry_2024-01-26.tif 60 of 120
conakry_2024-01-26.tif 80 of 120
conakry_2024-01-26.tif 100 of 120


Using cache found in C:\Users\ASUS/.cache\torch\hub\marccoru_marinedebrisdetector_main
Lightning automatically upgraded your loaded checkpoint from v1.8.2 to v2.6.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint D:\Guinea_Methodology\plasticwaste_experiments\https:\marinedebrisdetector.s3.eu-central-1.amazonaws.com\checkpoints\unet%2B%2B1\epoch=54-val_loss=0.50-auroc=0.987.ckpt`


## 3. Score it against the tide mask

In [5]:
!python 7_published_detector/02_score_predictions.py
pd.read_csv(ROOT / "7_published_detector/outputs/02_scores.csv").round(3)

         date  pixels_20m  patches  pixels_on_land  share_on_flats  box_share_flats
0  2024-01-11       78642      669            5012           0.063              0.1
1  2024-01-26       41467      420           62973           0.114              0.1
larger date over smaller: 1.9
overlap of the dates: 0.033


,date,pixels_20m,patches,pixels_on_land,share_on_flats,box_share_flats
0,2024-01-11,78642,669,5012,0.063,0.1
1,2024-01-26,41467,420,62973,0.114,0.1


**How to read it.** Compare `share_on_flats` with `box_share_flats`, and with the rejected forest: over 90 % on flats and a swing of 29 times between the dates. Near the box share, or as high as the forest, it fails on the same sediment. Then check the overlap: real debris moves, but a method that marks unrelated water on each date gives a list nobody can check. It would still produce candidates, because no confirmed debris exists for these waters.

## 4. MariNeXt, trained on MADOS

MADOS (Kikaki et al. 2024; 174 scenes, CC BY 4.0) has classes the other training sets lack: sediment-laden, turbid and shallow water, foam and waves. If the false detections on the flats come from sediment, a model that knows sediment should call the flats sediment, not debris. Its code is MIT-licensed and its five trained runs are public.

Its code needs the 2022 versions of PyTorch and mmcv. Here it runs on the GPU environment instead, with mmcv 1.6.0 as plain Python source kept outside it (`E:\SDG AI Lab\Guinea\marinext_test`). The compiled mmcv operations are not built; MariNeXt uses none of them, and all five runs load exactly.

Same two dates and tide mask as above. Debris is the class with the highest averaged probability over the five runs, so no threshold is chosen.

**Input.** MADOS was made from Acolite Rayleigh-corrected reflectance, and the scenes above are L2A surface reflectance. On open water, L2A sits 0.0 to 0.45 standard deviations from the MADOS band means, except B1, at 1.4 below. So MariNeXt is run twice: on L2A here, and on Acolite reflectance from the L1C scenes in 4b.

In [ ]:
GPU = r"E:\SDG AI Lab\Guinea\national_scan_2026\venv_gpu\Scripts\python"
MX = Path(r"E:\SDG AI Lab\Guinea\marinext_test")
# One-time setup, already done (about 100 MB on E:):
# !git clone --depth 1 https://github.com/gkakogeorgiou/mados.git "{MX}\repo"
# !{GPU} -m gdown --folder https://drive.google.com/drive/folders/1VwkFp47TEvRVXHNbucBmmylfZwIUmCWx -O "{MX}\weights"
# !{GPU} -m pip download --no-deps --no-binary :all: mmcv==1.6.0 -d "{MX}\tmp"     then copy its mmcv folder to {MX}\libs\mmcv,
# !{GPU} -m pip install --target "{MX}\libs" --no-deps addict yapf==0.40.1            and replace libs\mmcv\ops with the stub
#                                                                                    (any use of a compiled op raises)
print(sorted(p.parent.name + "/" + p.name for p in (MX / "weights").glob("*/*.pth")))
!{GPU} 7_published_detector/03_run_marinext.py 2>&1 | findstr /V /C:"Warning" /C:"warn"

About two minutes a date on the GPU. Then the same scoring as section 3, plus the class it gives the tidal flats:

In [ ]:
!{GPU} 7_published_detector/04_score_marinext.py
both = pd.concat([pd.read_csv(ROOT / "7_published_detector/outputs/02_scores.csv").assign(model="Russwurm et al. 2023"),
                  pd.read_csv(ROOT / "7_published_detector/outputs/04_scores_marinext.csv").assign(model="MariNeXt")])
both.set_index(["model", "date"]).round(3)

**Reading it.** MariNeXt calls most of the flats sediment-laden or open water, and under 1 % of flat pixels debris. So it does recognise the sediment. But the few debris pixels it gives are 96 % on flats, where flats are 10 % of the water. That fails the check in the same way as the rejected forest. The other detector does not.

It is steadier between the dates (1.7 times, overlap 0.30, against 0.03 for the other detector). Most of that steadiness comes from the flats, which do not move.

Neither result shows that any detection is debris.

### 4b. The same run on Acolite reflectance

The two L1C scenes (tile T28PFR, about 620 MB each) come from the Copernicus Data Space. The login is read from `C:\Users\ASUS\.cdse\credentials.json`, which stays on this computer. Acolite (Vanhellemont & Ruddick 2018; Windows release 20260421.0, GPL-3.0) then writes Rayleigh-corrected reflectance, and script 06 puts it on the L2A grid. Acolite could not fetch NASA ozone and pressure files without an Earthdata login, so it used standard values; at sea level this barely changes the Rayleigh correction.

Run in a terminal; about 15 minutes a scene, most of it on E:. On first use Acolite downloads its lookup tables.

On open water the Acolite scenes sit within 0.64 standard deviations of the MADOS band means in every band (B1: 0.26, against 1.4 on L2A). So the input now matches the training data.

In [ ]:
# Done once, in a terminal from ROOT:
#   & $GPU 7_published_detector/05_download_l1c.py      # L1C scenes to E:\SDG AI Lab\Guinea\marinext_test\l1c
#   & $GPU 7_published_detector/06_acolite_rhorc.py     # Acolite, then conakry_<date>_rhorc.tif in outputs
# Acolite: github.com/acolite/acolite/releases/download/20260421.0/acolite_py_win_20260421.0.tar.gz, unpacked to {MX}\acolite_bin
!{GPU} 7_published_detector/03_run_marinext.py rhorc 2>&1 | findstr /V /C:"Warning" /C:"warn"
!{GPU} 7_published_detector/04_score_marinext.py rhorc
runs = pd.concat([pd.read_csv(ROOT / "7_published_detector/outputs/04_scores_marinext.csv").assign(input="L2A"),
                  pd.read_csv(ROOT / "7_published_detector/outputs/04_scores_marinext_rhorc.csv").assign(input="Acolite rhorc")])
runs.set_index(["input", "date"]).round(3)

**Reading it.** With matching input the result holds: 94–95 % of MariNeXt's debris pixels lie on flats, against 10 % of the water. It again calls about half of the flats sediment-laden water. The swing between dates is 2.9 times, with an overlap of 0.21. So the failure on Guinea's flats comes from the model, not from how the scenes were prepared. MariNeXt is not adopted.